In [7]:
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import (train_test_split, learning_curve,
                                      validation_curve, cross_val_score)
from sklearn.metrics import mean_squared_error
import warnings; warnings.filterwarnings("ignore")

np.random.seed(42)

In [9]:
def bias_variance_demo(degree: int, n_samples: int = 50, n_experiments: int = 100):


    X_test_pts = np.array([0.5, 1.0, 1.5, 2.0, 2.5])
    true_vals   = np.sin(X_test_pts)

    all_preds = []
    for _ in range(n_experiments):
        X_train = np.random.uniform(0, np.pi, n_samples).reshape(-1,1)
        y_train = np.sin(X_train.ravel()) + np.random.normal(0, 0.2, n_samples)

        poly = Pipeline([
            ("poly", PolynomialFeatures(degree=degree, include_bias=False)),
            ("ridge", Ridge(alpha=1e-6)),
        ])
        poly.fit(X_train, y_train)
        preds = poly.predict(X_test_pts.reshape(-1,1))
        all_preds.append(preds)

    all_preds = np.array(all_preds)
    mean_pred  = all_preds.mean(axis=0)
    variance   = all_preds.var(axis=0).mean()
    bias_sq    = np.mean((mean_pred - true_vals)**2)
    return bias_sq, variance, bias_sq + variance

print("=== Bias-Variance Tradeoff (Polynomial Regression on sin(x)) ===")
print(f"{'Degree':>8} {'Bias²':>12} {'Variance':>12} {'Total Error':>14} {'Diagnosis':>18}")
for deg in [1, 2, 3, 5, 8, 12]:
    b, v, total = bias_variance_demo(deg)
    diagnosis = "High Bias (underfit)" if b > v*2 else "High Var (overfit)" if v > b*2 else "Balanced"
    print(f"  {deg:>6}   {b:>10.4f}   {v:>10.4f}   {total:>12.4f}   {diagnosis:>18}")


=== Bias-Variance Tradeoff (Polynomial Regression on sin(x)) ===
  Degree        Bias²     Variance    Total Error          Diagnosis
       1       0.0508       0.0046         0.0554   High Bias (underfit)
       2       0.0003       0.0017         0.0019   High Var (overfit)
       3       0.0002       0.0026         0.0028   High Var (overfit)
       5       0.0000       0.0043         0.0043   High Var (overfit)
       8       0.0000       0.0066         0.0067   High Var (overfit)
      12       0.0001       0.0083         0.0084   High Var (overfit)


In [10]:
from sklearn.datasets import make_regression
X_reg, y_reg = make_regression(n_samples=2000, n_features=15,
                                n_informative=8, noise=25, random_state=42)


model_simple  = Ridge(alpha=100.0)


model_complex = Pipeline([
    ("poly", PolynomialFeatures(degree=3, include_bias=False)),
    ("scaler", StandardScaler()),
    ("ridge", Ridge(alpha=0.01)),
])

print("=== Learning Curves (diagnose underfitting / overfitting) ===")
for label, model in [("Simple (Ridge α=100)", model_simple),
                     ("Complex (Poly degree=3)", model_complex)]:
    tr_sz, tr_sc, vl_sc = learning_curve(
        model, X_reg, y_reg,
        train_sizes=np.linspace(0.1,1.0,7),
        cv=5, scoring="neg_root_mean_squared_error",
        n_jobs=-1,
    )
    tr_rmse = -tr_sc.mean(axis=1)
    vl_rmse = -vl_sc.mean(axis=1)
    gap     = (vl_rmse - tr_rmse).mean()
    diagnosis = "High Bias (both high)" if vl_rmse.mean() > 60 else                 "High Variance (big gap)" if gap > 15 else "Good fit"

    print(f"{label}:")
    print(f"  Final train RMSE: {tr_rmse[-1]:.2f}  |  val RMSE: {vl_rmse[-1]:.2f}")
    print(f"  Gap (val-train):  {gap:.2f}  →  {diagnosis}")


=== Learning Curves (diagnose underfitting / overfitting) ===
Simple (Ridge α=100):
  Final train RMSE: 27.22  |  val RMSE: 27.46
  Gap (val-train):  1.39  →  Good fit
Complex (Poly degree=3):
  Final train RMSE: 17.41  |  val RMSE: 40.44
  Gap (val-train):  76.68  →  High Bias (both high)


In [11]:
X_reg2, y_reg2 = make_regression(n_samples=500, n_features=20,
                                  n_informative=5, noise=10, random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X_reg2, y_reg2, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_tr_s = scaler.fit_transform(X_tr)
X_te_s  = scaler.transform(X_te)

print("=== Regularisation Comparison ===")
print(f"{'Model':>20} {'Test RMSE':>12} {'Zero coeffs':>14} {'Non-zero':>12}")

for name, model in [
    ("OLS (no reg)",    Ridge(alpha=1e-6)),
    ("Ridge (α=10)",    Ridge(alpha=10)),
    ("Ridge (α=100)",   Ridge(alpha=100)),
    ("Lasso (α=0.1)",   Lasso(alpha=0.1)),
    ("Lasso (α=1.0)",   Lasso(alpha=1.0)),
    ("ElasticNet",       ElasticNet(alpha=0.1, l1_ratio=0.5)),
]:
    model.fit(X_tr_s, y_tr)
    rmse = np.sqrt(mean_squared_error(y_te, model.predict(X_te_s)))
    n_zero    = (np.abs(model.coef_) < 1e-4).sum()
    n_nonzero = (np.abs(model.coef_) >= 1e-4).sum()
    print(f"  {name:>18}   {rmse:>10.3f}   {n_zero:>12}   {n_nonzero:>10}")


=== Regularisation Comparison ===
               Model    Test RMSE    Zero coeffs     Non-zero
        OLS (no reg)        9.258              0           20
        Ridge (α=10)        9.220              0           20
       Ridge (α=100)       15.169              0           20
       Lasso (α=0.1)        9.174              6           14
       Lasso (α=1.0)        9.107             15            5
          ElasticNet        9.413              3           17


In [12]:
alphas = np.logspace(-3, 3, 15)
tr_sc, vl_sc = validation_curve(
    Ridge(), X_tr_s, y_tr,
    param_name="alpha", param_range=alphas,
    cv=5, scoring="neg_root_mean_squared_error", n_jobs=-1,
)
tr_rmse = -tr_sc.mean(axis=1)
vl_rmse = -vl_sc.mean(axis=1)
best_alpha = alphas[vl_rmse.argmin()]

print(f"=== Ridge Validation Curve ===")
print(f"Best α = {best_alpha:.4f}  (validation RMSE = {vl_rmse.min():.3f})")
print(f"{'Alpha':>10} {'Train RMSE':>14} {'Val RMSE':>12}")
for a, tr, vl in list(zip(alphas, tr_rmse, vl_rmse))[::3]:
    print(f"  {a:>8.4f}   {tr:>12.3f}   {vl:>10.3f}")

=== Ridge Validation Curve ===
Best α = 0.1389  (validation RMSE = 10.360)
     Alpha     Train RMSE     Val RMSE
    0.0010          9.788       10.360
    0.0193          9.788       10.360
    0.3728          9.788       10.360
    7.1969          9.912       10.483
  138.9495         23.388       24.290
